In [4]:
from pdb import set_trace as st
from pprint import pprint
import json
import subprocess
import sys

import msgspec
from tqdm.auto import tqdm
import pandas as pd
from pathlib import Path
import os
import re
import torch
from sentence_transformers import models, SentenceTransformer
from transformers import AutoTokenizer, AutoModel, HfArgumentParser
from tevatron.retriever.arguments import DataArguments, ModelArguments
from tevatron.retriever.arguments import TevatronTrainingArguments as TrainingArguments
from tevatron.retriever.modeling import DenseModel

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-base-en-v1.5")

encoder = msgspec.json.Encoder()
decoder = msgspec.json.Decoder()

/scratch/ft49/thuy0050/miniconda/conda/envs/tevatron/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Utils


In [29]:
def read_json(file_path, jsonl=False):
    file_path = Path(file_path)
    if not file_path.is_file():
        raise ValueError("filepath is not a file")
    # if not file_path.suffix == ".jsonl" and jsonl:
    #     raise ValueError("the file is not jsonl")
    # if not file_path.suffix == ".json" and not jsonl:
    #     raise ValueError("the file is not json")
    file_path = file_path.__str__()

    with open(file_path, "rb") as file:
        data = file.read()
    if jsonl:
        output = decoder.decode_lines(data)
    else:
        output = decoder.decode(data)

    print(f"The file is of type: {type(output)}")
    print(f"The file contains {len(output)} items.")
    return output


def write_json(file_path, data, jsonl=False):
    with open(file_path, "wb") as file:
        if jsonl:
            file.write(encoder.encode_lines(data))
        else:
            file.write(encoder.encode(data))
    print(f"The file contains {len(data)} items.")
    print("Saved to", file_path)


def read_tsv(file_path, row_names=None):
    file_path = Path(file_path)
    if file_path.is_file() and file_path.suffix == ".tsv" :
        temp = pd.read_csv(file_path, sep='\t', names=row_names)
    else:
        raise ValueError("filepath is not a file or it is not a tsv file.")
    return temp


# Compute token embeddings
def get_sentence_embeddings(data_args, model, tokenizer, q=None, p=None):
    
    if q is not None:
        q = tokenizer(
            q,
            padding=True,
            truncation=True,
            max_length=(
                data_args.query_max_len - 1
                if data_args.append_eos_token
                else data_args.query_max_len
            ),
            pad_to_multiple_of=data_args.pad_to_multiple_of,
            return_attention_mask=True,
            return_tensors="pt",
            return_token_type_ids=False,
            add_special_tokens=True,
        )
        for k, v in q.items():
            q[k] = v.to("cuda")
    elif p is not None:
        p = tokenizer(
            p,
            padding=True,
            truncation=True,
            max_length=(
                data_args.passage_max_len - 1
                if data_args.append_eos_token
                else data_args.passage_max_len
            ),
            pad_to_multiple_of=data_args.pad_to_multiple_of,
            return_attention_mask=True,
            return_tensors="pt",
            return_token_type_ids=False,
            add_special_tokens=True,
        )
        for k, v in p.items():
            p[k] = v.to("cuda")
    
    model.eval()
    with torch.amp.autocast("cuda", dtype=torch.bfloat16):
        with torch.no_grad():
            if q is not None:
                output = model(query=q).q_reps
            else:
                output = model(passage=p).p_reps
    return output


def get_cosine_similarity(a, b, normalize=True):
    # This assume that the embedding has been normalized
    
    if normalize:
        a = torch.nn.functional.normalize(a, p=2, dim=1)
        b = torch.nn.functional.normalize(b, p=2, dim=1)
    
    return torch.matmul(a, b.T)

In [86]:
# bsz : batch size (number of positive pairs)
# d   : latent dim
# x   : Tensor, shape=[bsz, d]
#       latents for one side of positive pairs
# y   : Tensor, shape=[bsz, d]
#       latents for the other side of positive pairs

def align_loss(x, y, alpha=2):
    return (x - y).norm(p=2, dim=1).pow(alpha).mean()

def uniform_loss(x, t=2):
    return torch.pdist(x, p=2).pow(2).mul(-t).exp().mean().log()

# Check temporal embeddings

## Model 1

### 1. Params

In [54]:
DATA_ROOT_DIR="/home/thuy0050/mg61_scratch2/thuy0050/data/third_work"
OUTPUT_DIR_ROOT="/home/thuy0050/mg61_scratch2/thuy0050/exp/tevatron"

DATA_NAME="temporal_nobel_prize"
MODEL_NAME="ts-retriever"
BACKBONE="contriever"
EXP_NAME="v3_qt_bs64"
OUTPUT_DIR=f"{OUTPUT_DIR_ROOT}/{DATA_NAME}/{MODEL_NAME}/{BACKBONE}/{EXP_NAME}"

CHECKPOINT_DIR=OUTPUT_DIR

sys.argv = [
    "train_tsretriever_with_temporal_v4.py",  # dummy script name
    "--pooling", "avg",
    "--bf16",
    "--normalize",
    "--query_max_len", "512",
    "--passage_max_len", "512",
    "--attn_implementation", "sdpa",
    "--lora",
    "--lora_r", "4",
    "--lora_alpha", "16",
    "--lora_target_modules", "all-linear",
    "--modules_to_save", "temporal_projector",
    "--dataset_name", f"{DATA_ROOT_DIR}/tevatron/Tevatron___msmarco-passage",
    "--dataset_path", f"{DATA_ROOT_DIR}/temporal/temporal_nobel_prize/train/train_temporal_v2.jsonl",
    "--eval_dataset_path", f"{DATA_ROOT_DIR}/temporal/temporal_nobel_prize/train/dev.jsonl",
    "--model_name_or_path", CHECKPOINT_DIR,
    "--run_name", f"{BACKBONE}_{EXP_NAME}"
]

### Load the model

In [55]:
torch_dtype = torch.bfloat16

parser = HfArgumentParser((ModelArguments, DataArguments, TrainingArguments))

model_args, data_args, training_args = parser.parse_args_into_dataclasses()
model_args: ModelArguments
data_args: DataArguments
training_args: TrainingArguments

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

if data_args.padding_side == 'right':
    tokenizer.padding_side = 'right'
else:
    tokenizer.padding_side = 'left'
    
model1 = DenseModel.load(
    model_args.model_name_or_path,
    pooling=model_args.pooling,
    normalize=model_args.normalize,
    lora_name_or_path=model_args.lora_name_or_path,
    cache_dir=model_args.cache_dir,
    torch_dtype=torch_dtype,
    attn_implementation=model_args.attn_implementation,
)
model1.to("cuda")
# for k in model.state_dict().keys():
#     print(k)

BertModel does not support Flash Attention 2.0 yet. Please request to add support where the model is hosted, on its model hub page: https://huggingface.co/facebook/contriever/discussions/new or in the Transformers GitHub repo: https://github.com/huggingface/transformers/issues/new
Fall back to use sdpa
Please provide lora_name_or_path to load the PEFT model correctly!!!


DenseModel(
  (encoder): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): lora.Linear(
                (base_layer): Linear(in_features=768, out_features=768, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=768, out_features=4, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=4, out_featur

## Model 2

### 1. Params

In [56]:
DATA_ROOT_DIR="/home/thuy0050/mg61_scratch2/thuy0050/data/third_work"
OUTPUT_DIR_ROOT="/home/thuy0050/mg61_scratch2/thuy0050/exp/tevatron"

DATA_NAME="temporal_nobel_prize"
MODEL_NAME="ts-retriever"
BACKBONE="contriever"
EXP_NAME="v4_qt_reconstruction_random"
OUTPUT_DIR=f"{OUTPUT_DIR_ROOT}/{DATA_NAME}/{MODEL_NAME}/{BACKBONE}/{EXP_NAME}"

CHECKPOINT_DIR=OUTPUT_DIR

sys.argv = [
    "train_tsretriever_with_temporal_v4.py",  # dummy script name
    "--pooling", "avg",
    "--bf16",
    "--normalize",
    "--query_max_len", "512",
    "--passage_max_len", "512",
    "--attn_implementation", "sdpa",
    "--lora",
    "--lora_r", "4",
    "--lora_alpha", "16",
    "--lora_target_modules", "all-linear",
    "--modules_to_save", "temporal_projector",
    "--dataset_name", f"{DATA_ROOT_DIR}/tevatron/Tevatron___msmarco-passage",
    "--dataset_path", f"{DATA_ROOT_DIR}/temporal/temporal_nobel_prize/train/train_temporal_v2.jsonl",
    "--eval_dataset_path", f"{DATA_ROOT_DIR}/temporal/temporal_nobel_prize/train/dev.jsonl",
    "--model_name_or_path", CHECKPOINT_DIR,
    "--run_name", f"{BACKBONE}_{EXP_NAME}"
]

### 2. Load the model

In [57]:
torch_dtype = torch.bfloat16

parser = HfArgumentParser((ModelArguments, DataArguments, TrainingArguments))

model_args, data_args, training_args = parser.parse_args_into_dataclasses()
model_args: ModelArguments
data_args: DataArguments
training_args: TrainingArguments

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

if data_args.padding_side == 'right':
    tokenizer.padding_side = 'right'
else:
    tokenizer.padding_side = 'left'
    
model2 = DenseModel.load(
    model_args.model_name_or_path,
    pooling=model_args.pooling,
    normalize=model_args.normalize,
    lora_name_or_path=model_args.lora_name_or_path,
    cache_dir=model_args.cache_dir,
    torch_dtype=torch_dtype,
    attn_implementation=model_args.attn_implementation,
)
model2.to("cuda")
# for k in model.state_dict().keys():
#     print(k)

BertModel does not support Flash Attention 2.0 yet. Please request to add support where the model is hosted, on its model hub page: https://huggingface.co/facebook/contriever/discussions/new or in the Transformers GitHub repo: https://github.com/huggingface/transformers/issues/new
Fall back to use sdpa
Please provide lora_name_or_path to load the PEFT model correctly!!!


DenseModel(
  (encoder): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): lora.Linear(
                (base_layer): Linear(in_features=768, out_features=768, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=768, out_features=4, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=4, out_featur

# Analysis

In [ ]:
prompt = "Represent this sentence for searching relevant passages: "

q_pos_text = prompt + "Jermaine Beckford played for which team from 2003 to 2004?"
qt_pos_text = prompt + "from 2003 to 2004?"

q_neg_text = prompt + "Jermaine Beckford played for which team from 1996 to 2002?"
qt_neg_text = prompt + "from 1996 to 2002"

p_text = "Beckford originally began his career in the Chelsea youth team , coming through the schoolboy ranks at the same time as Carlton Cole . Rejected by Chelsea in 2003 , he was signed up by Wealdstone , then in the Isthmian Premier League , and played as a semi-professional for three years whilst also working as a windscreen fitter for the RAC . His very impressive goal scoring record for Wealdstone attracted a lot of attention from Football League sides and reportedly more than 30 professional clubs showed an interest in the prolific striker , with many sending scouts to watch him play for Wealdstone . He had a trial with Championship side Crystal Palace , before signing for Leeds United in March 2006 for an undisclosed fee , having scored 35 goals in 40 games for Wealdstone that season ."

In [87]:
p = get_sentence_embeddings(data_args, model1, tokenizer, p=[p_text])
q_pos = get_sentence_embeddings(data_args, model1, tokenizer, q=[q_pos_text])
q_neg = get_sentence_embeddings(data_args, model1, tokenizer, q=[q_neg_text])
print("sim(q+, p)", get_cosine_similarity(q_pos, p))
print("sim(q-, p)", get_cosine_similarity(q_neg, p))

print(f"alignment q+ p: {align_loss(q_pos, p)}\nalignment q- p: {align_loss(q_neg, p)}\nuniformity: {uniform_loss(torch.concat([p, q_pos, q_neg], dim=0))}")

sim(q+, p) tensor([[0.6903]], device='cuda:0')
sim(q-, p) tensor([[0.5274]], device='cuda:0')
alignment q+ p: 0.6194626092910767
alignment q- p: 0.9451271295547485
uniformity: -1.1145455837249756


/scratch/ft49/thuy0050/miniconda/conda/envs/tevatron/lib/python3.10/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [88]:
p = get_sentence_embeddings(data_args, model2, tokenizer, p=[p_text])
q_pos = get_sentence_embeddings(data_args, model2, tokenizer, q=[q_pos_text])
q_neg = get_sentence_embeddings(data_args, model2, tokenizer, q=[q_neg_text])
print("sim(q+, p)", get_cosine_similarity(q_pos, p))
print("sim(q-, p)", get_cosine_similarity(q_neg, p))

print(f"alignment q+ p: {align_loss(q_pos, p)}\nalignment q- p: {align_loss(q_neg, p)}\nuniformity: {uniform_loss(torch.concat([p, q_pos, q_neg], dim=0))}")

sim(q+, p) tensor([[0.7028]], device='cuda:0')
sim(q-, p) tensor([[0.5539]], device='cuda:0')
alignment q+ p: 0.5944311022758484
alignment q- p: 0.892104983329773
uniformity: -1.0550647974014282


alignment q+ p: 0.5944311022758484
alignment q- p: 0.892104983329773
uniformity: -1.0550647974014282
